In [1]:
import argparse
from itertools import product

import numpy as np

from cc2cc import add_args, cc, ucc
from cc2cc.utils import Grid, gen_mole, print_computer_info
from cc2cc.utils.env_var import DATA_PATH
from cc2cc.utils.parser import gen_name_args

origin_mol_str_list = [
    # "molecule0-W4_11",
    # "molecule1-W4_11",
    # "molecule2-W4_11",
    # "molecule3-W4_11",
    # "molecule4-W4_11",
    # "molecule5-W4_11",
]
name_mol_str_list = [
    # "molecule0",
    # "molecule1",
    # "molecule2",
    # "molecule3",
    # "molecule4",
    # "molecule5",
    "molecule6",
]
name_mol_str_exclude_list = [
]

mol_elements_dict = {}
len_elements_dict = {}

if __name__ == "__main__":
    error_molecule = []

    name_mol_list = gen_name_args(name_mol_str_list, "gmtkn-def2")
    name_mol_exclude_list = gen_name_args(
        name_mol_str_exclude_list, "gmtkn-def2", if_exclude=True
    )
    name_mol_list = [mol for mol in name_mol_list if mol not in name_mol_exclude_list]

    origin_mol_list = gen_name_args(origin_mol_str_list, "gmtkn-def2")
    name_mol_list = origin_mol_list + name_mol_list

    error_molecule = []
    print(f"Name Molecule List: {name_mol_list}")

    for name_mol in name_mol_list:
        try:
            mol = gen_mole(
                name_mol,
                0,
                1,
                0,
                "def2-TZVPD",
                "gmtkn-def2",
                if_rotate=True,
                if_rotate_random=False,
                solve_symmetry=True,
                verbose=1,
            )

            mol_elements = list(np.array(mol.elements))
            mol_atom_coords = list(mol.atom_coords())
            mol_atom_coords.append(name_mol)
            # print(mol_atom_coords)
            mol_elements.extend([mol.charge, mol.spin])
            mol_elements_str = "-".join(map(str, mol_elements))
            if mol_elements_str not in mol_elements_dict:
                mol_elements_dict[mol_elements_str] = [mol_atom_coords]
            else:
                mol_elements_dict[mol_elements_str].append(mol_atom_coords)
            len_elements_dict[mol_elements_str] = len(list(np.array(mol.elements)))

        except (ValueError, RuntimeError) as e:
            print(f"ERROR: {name_mol}")
            print(e)
            error_molecule.append(name_mol)
            print(f"Error molecule: {error_molecule}")
        finally:
            print(f"Processed: {name_mol}")
        print()

    print(f"Error molecule: {error_molecule}")

Name Molecule List: ['RG18-ne6', 'ALK8-li5_ch', 'IL16-144B', 'PArel-c2cl41', 'PArel-c2cl42', 'PArel-c2cl43', 'RSE43-P28', 'HAL59-NH3_F3CI', 'PArel-c2h2f41', 'PArel-c2h2f42', 'RSE43-E28', 'ISO34-E19', 'ISO34-E20', 'ISO34-P20', 'PNICO23-11', 'PNICO23-12', 'PNICO23-22', 'S22-03', 'S22-12a', 'YBDE18-f2s-cbh22', 'HAL59-pyr', 'RC21-10e', 'S66-18B', 'S66-19B', 'S66-25A', 'S66-25B', 'S66-27B', 'S66-29A', 'S66-33A', 'S66-48A', 'S66-48B', 'S66-49B', 'S66-58A', 'S66-58B', 'S66-65A', 'S66-66B', 'BHDIV10-ed2', 'BHDIV10-ed8', 'BHDIV10-ts2', 'BHDIV10-ts8', 'CHB6-25B', 'CHB6-26B', 'CHB6-27B', 'DC13-C6H6', 'FH51-methylfuran', 'FH51-methylimidazole', 'FH51-methylpyrazole', 'G2RC-94', 'HAL59-11_benF3-benB', 'HAL59-12_benF6-benB', 'HAL59-27_CH3Br-benA', 'HAL59-28_CH3I-benA', 'HAL59-29_CF3Br-benA', 'HAL59-30_CF3I-benA', 'ICONF-N3P3H12_1', 'ICONF-N3P3H12_2', 'ISO34-E31', 'ISO34-P31', 'NBPRC-bz', 'PNICO23-10', 'PX13-hf_6', 'PX13-hf_6_ts', 'RC21-7e', 'RG18-bz', 'S22-04', 'S22-10a', 'S22-11a', 'S22-14b', 'S22-

In [2]:
for mol_elements_name, mol_elements in mol_elements_dict.items():
    print(f"Processing {len_elements_dict[mol_elements_name]}")
    # print(f"{mol_elements}")

    identifiables = [0]
    for i_elements in range(1, len(mol_elements)):
        distance_list = np.zeros(len(identifiables))
        for iter, identifiable in enumerate(identifiables):
            for i_element in range(len(mol_elements[i_elements]) - 1):
                distance_list[iter] = max(
                    np.linalg.norm(
                        mol_elements[i_elements][i_element]
                        - mol_elements[identifiable][i_element]
                    ),
                    distance_list[iter],
                )
        if np.all(distance_list > 0.2):
            identifiables.append(i_elements)
        # else:
        #     print(f"Skipping {mol_elements[i_elements][-1]}")

    # print("===identifiables===")
    for identifiable in identifiables:
        if mol_elements[identifiable][-1].startswith("W4_11"):
            continue
        print(f'"{mol_elements[identifiable][-1]}",')
        # print(
        #     f"{np.array2string(np.array(mol_elements[identifiable][:-1]), formatter={'float': '{: .2f}'.format})}"
        # )
    print()

Processing 6
"RG18-ne6",

Processing 7
"ALK8-li5_ch",

Processing 7
"IL16-144B",

Processing 7
"PArel-c2cl41",
"PArel-c2cl42",
"PArel-c2cl43",

Processing 8
"RSE43-P28",

Processing 9
"HAL59-NH3_F3CI",

Processing 9
"PArel-c2h2f41",
"PArel-c2h2f42",

Processing 9
"RSE43-E28",

Processing 10
"ISO34-E19",
"ISO34-E20",
"ISO34-P20",

Processing 10
"PNICO23-11",
"PNICO23-12",

Processing 10
"PNICO23-22",

Processing 10
"S22-03",

Processing 10
"YBDE18-f2s-cbh22",

Processing 11
"HAL59-pyr",
"S66-25A",
"S66-33A",
"S66-58A",
"S66-58B",

Processing 11
"RC21-10e",

Processing 12
"BHDIV10-ed2",
"BHDIV10-ts2",

Processing 12
"BHDIV10-ed8",
"BHDIV10-ts8",
"CHB6-25B",
"G2RC-94",
"HAL59-11_benF3-benB",
"HAL59-27_CH3Br-benA",
"HAL59-28_CH3I-benA",
"HAL59-29_CF3Br-benA",
"S22-11a",
"S66-24A",
"S66-28A",
"S66-47A",
"S66-47B",

Processing 12
"FH51-methylfuran",

Processing 12
"FH51-methylimidazole",
"FH51-methylpyrazole",
"TAUT15-8a",
"TAUT15-8b",

Processing 12
"ICONF-N3P3H12_1",
"ICONF-N3P3H12_2",

Pr

In [8]:
mol_elements_dict

{'Be-Cl-Cl-0-0': [[array([0., 0., 0.]),
   array([-3.40120467,  0.        ,  0.        ]),
   array([3.40120467, 0.        , 0.        ]),
   'W4_11-becl2']],
 'Be-F-F-0-0': [[array([0., 0., 0.]),
   array([-2.60704726,  0.        ,  0.        ]),
   array([2.60704726, 0.        , 0.        ]),
   'W4_11-bef2']],
 'C-Cl-Cl-0-0': [[array([0.       , 1.6089375, 0.       ]),
   array([-2.64638569, -0.27252296,  0.        ]),
   array([ 2.64638569, -0.27252296,  0.        ]),
   'W4_11-ccl2']],
 'C-F-F-0-0': [[array([ 0.        , -1.13580973,  0.        ]),
   array([-1.94261956,  0.35899387,  0.        ]),
   array([1.94261956, 0.35899387, 0.        ]),
   'W4_11-cf2']],
 'O-Cl-Cl-0-0': [[array([ 0.        , -1.48572494,  0.        ]),
   array([-2.64783133,  0.33525673,  0.        ]),
   array([2.64783133, 0.33525673, 0.        ]),
   'W4_11-cl2o']],
 'C-N-Cl-0-0': [[array([1.28237933, 0.        , 0.        ]),
   array([3.47820707, 0.        , 0.        ]),
   array([-1.80865703,  0.   